# S-Fig 8 — Within-Subject Variance Violins

Distribution of within-subject prediction std(prob) for correct vs incorrect subjects.  
**Source**: collected prediction parquets  
**Head**: Transformer  
**Tasks**: all tasks with parquets available

In [ ]:
import sys
from pathlib import Path

# ── Workspace root (parent of NSRR-tools/) ────────────────────────────────────
WORKSPACE_ROOT = Path("../../../../..").resolve()   # adjust if notebook depth differs
NSRR_TOOLS     = WORKSPACE_ROOT / "NSRR-tools"
FINAL_RESULTS  = WORKSPACE_ROOT / "final_results"
PAPER_FIGURES  = NSRR_TOOLS / "results" / "paper_figures"
FINAL_OUT      = PAPER_FIGURES / "final"
FINAL_OUT.mkdir(parents=True, exist_ok=True)

# Add utils to path
sys.path.insert(0, str(Path(".").resolve()))

from utils.style import (
    apply_tbme_style, save_figure, FULL_W, HALF_W,
    MAIN_TASKS, SUPP_TASKS, ALL_TASKS, BINARY_MAIN,
    HEAD_STYLE, TASK_LABEL,
)
from utils.data import set_root, load_analysis, load_heatmap, load_parquets
from utils import panels

import matplotlib
matplotlib.use("Agg")   # comment out in Jupyter to get inline plots
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

set_root(WORKSPACE_ROOT)
apply_tbme_style()
print("Setup OK — workspace root:", WORKSPACE_ROOT)

In [ ]:
# Panel labeling helper
def add_panel_label(ax, label, x=0.02, y=0.97):
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=8, fontweight="bold", va="top", fontfamily="serif")

In [ ]:
HEAD   = "transformer"
SPLIT  = "test"
TASKS  = ALL_TASKS
SHOW_CONTEXTS = ["30s", "40m", "120m", "240m"]
N_COLS = 4
N_ROWS = (len(TASKS) + N_COLS - 1) // N_COLS

# Load parquets (only contexts we care about)
pqs = {}
for task in TASKS:
    p = load_parquets("phase0_v3", task, HEAD, SPLIT)
    if p:
        pqs[task] = p
print("Tasks with parquets:", sorted(pqs.keys()))

In [ ]:
fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(FULL_W, N_ROWS * 2.2))
axes_flat = axes.flatten()

plot_tasks = [t for t in TASKS if t in pqs]
for i, (ax, task) in enumerate(zip(axes_flat, plot_tasks)):
    panels.variance_violin_panel(ax, pqs[task], contexts=SHOW_CONTEXTS)
    ax.set_title(TASK_LABEL.get(task, task), fontsize=8)
    add_panel_label(ax, f"({chr(97+i)})")
    if i > 0 and ax.get_legend():
        ax.get_legend().remove()

handles, labels = axes_flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, fontsize=7,
           bbox_to_anchor=(0.5, 1.02), frameon=False)

for ax in axes_flat[len(plot_tasks):]:
    ax.set_visible(False)

fig.tight_layout(rect=[0, 0, 1, 0.97], h_pad=1.5, w_pad=1.0)
save_figure(fig, FINAL_OUT, "sfig8_variance_violins")
plt.show()